In [ ]:
# import packages
import numpy as np
import pandas as pd
import seaborn as sb
import glob 

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-pastel')

Step 1: merge all the participant files from 1C

In [ ]:
folder_path = "dfs_per_participant"
files = glob.glob(f"{folder_path}/*csv") #get name of all files in folder

dataframes = [] #list of all dataframes
for file in files:
    print('loading:', file)
    df = pd.read_csv(file)
    dataframes.append(df)
full_df = pd.concat(dataframes, ignore_index=True) #merge the dataframes


Step 2; create a classifications target

first attempt:
for avg_mood_target
•  low: mood < 5 
•  medium: 5 ≤ mood < 7 
•  high: mood ≥ 7
This is thus for the average mood over that timespan

second attempt:
Due to strong class imbalance when using fixed thresholds, we applied a quantile-based discretization to ensure balanced class distributions, improving the reliability of the classification models. groups aren’t perfectly equal because qcut can’t split identical values and must keep them together. cutoff values can be seen from the classification action. Low:2.999 - 6.8 < medium: 6.8 - 7.4 < high: 7.4 - 9.333.



In [ ]:
full_df["mood_class"] = pd.qcut(full_df["avg_mood_target"], q=3, labels=["low", "medium", "high"]) #split the mood values into 3 equally sized groups and label them low, medium, high basef on the avg_mood_target values.
cutoff_values = pd.qcut(full_df["avg_mood_target"], q=3) #gets the cutoff values
full_df.to_csv('random forest datasets/dataset_classification.csv', index=False)

Step 3; split train/test set in a time-aware way
Since our rows are sequential windows per patient, the split should respect time and avoid training on future rows while testing on earlier ones. What “time-aware split” means here. For each participant separately: take the earlier rows for training, take the later rows for testing. Then do that for every participant, and combine all participant-level train parts into one training set and all test parts into one test set.

In [ ]:
train_parts = [] #early data
test_parts = [] #later data
split_value = 0.8

for partcipant_id, group in full_df.groupby('id'):  #split dataset into groups based on id
    group = group.sort_values('target_day') #sort by target day
    split_idx = int(len(group) * split_value) #decide where to split
    
    train_group = group.iloc[:split_idx]
    test_group = group.iloc[split_idx:]

    train_parts.append(train_group)
    test_parts.append(test_group)

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

step 4; removing target leakage columns.
Target leakage = giving the model information it should not have at prediction time. As we try to predict next-day mood, the model should only see information from the past (window). Not anything from the target day. therefore we remove the target features from the dataframe, so we remove avg_mood_target, std_mood_target, trend_mood_target from the features. Additionally we remove id, t, and target_day as features.

In [ ]:
cols_to_drop = [
    "mood_class",          # target
    "avg_mood_target",     # leakage
    "std_mood_target",     # leakage
    "trend_mood_target",   # leakage
    "id",                  # not a feature
    "t",                   # not useful
    "target_day"           # future info
]

X_train = train_df.drop(columns=cols_to_drop) #training features
y_train = train_df["mood_class"]

X_test = test_df.drop(columns=cols_to_drop) #same as training features
y_test = test_df["mood_class"]


Train a first random forest:


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42) #to make the results reproducible
rf.fit(X_train, y_train)

make predictions

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred)) #gives the overall percentage of correct predictions
print(classification_report(y_test, y_pred)) #gives more detail for each class
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {"n_estimators": [100, 200], "max_depth": [5, 10, None],}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3, #Use 3-fold cross-validation
    scoring="f1_weighted",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))